<h2>Extract and preprocess meaningful features from unstructured text attributes using NLP techniques such as tokenization, TF-IDF vectorization, and sentiment analysis to integrate textual insights into structured datasets.</h2>

In [5]:
import pandas as pd

In [6]:
data = {
    'customer_id': [101, 102, 103, 104, 105],
    'review_text': [
        "Great product! Fast shipping and excellent quality.",
        "Terrible service. The item arrived damaged and late.",
        "Average experience, nothing special but works fine.",
        "Loved it! Highly recommend to everyone.",
        "Poor quality material. Completely dissatisfied!"
    ]
}

In [7]:
df = pd.DataFrame(data)

In [8]:
print("Original Dataset:")
print(df)

Original Dataset:
   customer_id                                        review_text
0          101  Great product! Fast shipping and excellent qua...
1          102  Terrible service. The item arrived damaged and...
2          103  Average experience, nothing special but works ...
3          104            Loved it! Highly recommend to everyone.
4          105    Poor quality material. Completely dissatisfied!


In [9]:
import re

In [10]:
def clean_and_tokenize(text):
    text = text.lower()  
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    tokens = text.split()  
    return tokens

In [11]:
df['clean_tokens'] = df['review_text'].apply(clean_and_tokenize)

In [12]:
df['clean_text'] = df['clean_tokens'].apply(lambda tokens: ' '.join(tokens))

In [13]:
df['word_count'] = df['clean_tokens'].apply(len)

In [14]:
print("Preprocessed Text & Basic Metrics:")
print(df[['review_text', 'clean_text', 'word_count']].head())

Preprocessed Text & Basic Metrics:
                                         review_text  \
0  Great product! Fast shipping and excellent qua...   
1  Terrible service. The item arrived damaged and...   
2  Average experience, nothing special but works ...   
3            Loved it! Highly recommend to everyone.   
4    Poor quality material. Completely dissatisfied!   

                                          clean_text  word_count  
0  great product fast shipping and excellent quality           7  
1  terrible service the item arrived damaged and ...           8  
2  average experience nothing special but works fine           7  
3              loved it highly recommend to everyone           6  
4      poor quality material completely dissatisfied           5  


In [15]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [16]:
nltk.download('vader_lexicon', quiet=True)

True

In [17]:
sid = SentimentIntensityAnalyzer()

In [18]:
df['sentiment_score'] = df['clean_text'].apply(lambda x: sid.polarity_scores(x)['compound'])

In [19]:
print("Extracted Sentiment Scores:")
print(df[['review_text', 'sentiment_score']].head())

Extracted Sentiment Scores:
                                         review_text  sentiment_score
0  Great product! Fast shipping and excellent qua...           0.8316
1  Terrible service. The item arrived damaged and...          -0.7184
2  Average experience, nothing special but works ...           0.1459
3            Loved it! Highly recommend to everyone.           0.7713
4    Poor quality material. Completely dissatisfied!          -0.7178


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [22]:
tfidf = TfidfVectorizer(max_features=5, stop_words='english')

In [23]:
tfidf_matrix = tfidf.fit_transform(df['clean_text']).toarray()

In [24]:
tfidf_df = pd.DataFrame(
    tfidf_matrix, 
    columns=[f'tfidf_{word}' for word in tfidf.get_feature_names_out()]
)

In [25]:
df_final = pd.concat([df[['customer_id', 'word_count', 'sentiment_score']], tfidf_df], axis=1)

In [26]:
print("Final Structured Dataset with Text Features:")
print(df_final)

Final Structured Dataset with Text Features:
   customer_id  word_count  sentiment_score  tfidf_arrived  tfidf_average  \
0          101           7           0.8316            0.0            0.0   
1          102           8          -0.7184            1.0            0.0   
2          103           7           0.1459            0.0            1.0   
3          104           6           0.7713            0.0            0.0   
4          105           5          -0.7178            0.0            0.0   

   tfidf_completely  tfidf_dissatisfied  tfidf_quality  
0          0.000000            0.000000       1.000000  
1          0.000000            0.000000       0.000000  
2          0.000000            0.000000       0.000000  
3          0.000000            0.000000       0.000000  
4          0.614189            0.614189       0.495524  
